<a href="https://colab.research.google.com/github/Eliezer-Carvalho/Adamastor-GPT/blob/main/Adamastor_GPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **Adamastor-GPT**

Este modelo é baseado na arquitetura <b> Transformer </b>, inspirado nos modelos do tipo <b> GPT (<i>Generative Pre-trained Transformer</i>)</b>. <br>

Ao longo deste Colab, iremos navegar pelas componentes cruciais dos modelos GPT-like, que atualmente dominam grande parte do mundo da Inteligência Artificial.

<b> No final, teremos um modelo estilo GPT funcional, capaz de realizar inferência de forma limpa e consistente. </b>

In [2]:
from tokenizers import Tokenizer
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

device = "cuda" if torch.cuda.is_available() else "cpu" #Mover tudo para a GPU #Fulcral para treino

### Dados do Modelo

In [3]:
Sequence_Length = 32
Batch_Size = 16
Embedding_Dimension = 64
Head_Size = 8 #Head_Size = Embedding_Dimension // Number_Head
Number_Head = 8

###


---

### Tokenization


In [5]:
Tokenizer = Tokenizer.from_file ("GAMA")

with open ("Os Lusiadas.txt", "r", encoding = "utf-8") as f:
  dataset = f.read()

Tokenization = Tokenizer.encode (dataset)
print (len(Tokenization.ids))
print (len(Tokenization.tokens))

tensor = torch.tensor (Tokenization.ids, dtype = torch.long)
print (tensor [:50])

n = int (0.9 * (len(tensor)))
dados_treino = tensor [:n] #90% Dados de Treino
dados_teste = tensor [n:] #10% Dados de Teste

74830
74830
tensor([  26, 4359,   23,   32,   30,   69,   13,   16,   13, 4359,    0,   23,
        1526,   94,  679,  119,  527,    0, 2146,   21,    2,    0,  342,  670,
        1060, 2561,  527, 2459,  396,  142,  137, 2852,  114,  206, 4180,  781,
        3292, 2513,  573, 4572,  692,  396, 1997,  660,   38,  541,  113,  362,
         137,   31])


###


---

### Batch e Sequence Length

In [15]:
torch.manual_seed (2000)

def Batch ():
  #dados = train_data if split == 'treino' else training_dats
  idx = torch.randint (0, len(dados_treino) - Sequence_Length, (Batch_Size, ))
  x = torch.stack ([dados_treino [i: i + Sequence_Length] for i in idx])
  y = torch.stack ([dados_treino [i + 1: i + Sequence_Length + 1] for i in idx])

  x, y = x.to(device), y.to(device) #Passa dados para a GPU

  return x, y

x, y = Batch()

print (x.shape) #(B, T)
print (x[1])

print (y.shape) #(B, T)
print (y[1])

torch.Size([16, 32])
tensor([1061,   89,  150,  108,   27, 4417, 1116, 4659, 2880,  105,  727, 4563,
         144,  395,  115,  147, 1254, 2778,  272,  353,  472, 1611, 2883,  224,
         330, 1334,    0, 2560, 1186,  604,  343,  728])
torch.Size([16, 32])
tensor([  89,  150,  108,   27, 4417, 1116, 4659, 2880,  105,  727, 4563,  144,
         395,  115,  147, 1254, 2778,  272,  353,  472, 1611, 2883,  224,  330,
        1334,    0, 2560, 1186,  604,  343,  728,  763])


###


---



### Definição de uma Head

In [10]:
class Head (nn.Module): #Head = Attention
  def __init__(self):
    super().__init__()
    self.Query = nn.Linear (Embedding_Dimension, Head_Size, bias = False)
    self.Key = nn.Linear (Embedding_Dimension, Head_Size, bias = False)
    self.Value = nn.Linear (Embedding_Dimension, Head_Size, bias = False)

  def forward (self, x):
    B, T, C = x.shape

    Q = self.Query (x)
    K = self.Key (x)
    V = self.Value (x)

    attention_score = Q @ K.transpose (-2, -1) * (Head_Size ** -0.5)

    tril = torch.tril (torch.ones (T, T, device = x.device))
    attention_score = attention_score.masked_fill (tril == 0, float ("-inf"))

    attention_score = F.softmax (attention_score, dim = -1)

    output = attention_score @ V
    return output


###


---


### Definição de Multi Head

In [11]:
class Multi_Head (nn.Module):
  def __init__(self):
    super().__init__()
    self.Concat = nn.ModuleList ([Head() for i in range (Number_Head)]) #Head Concatenation #Definição de Multi Head Attention
    self.Output_Projection = nn.Linear (Embedding_Dimension, Embedding_Dimension) #Necessário para misturar a informação das Heads #Desta forma volta também à dimensão (B, T, C)

  def forward (self, x):

    output = self.Output_Projection (torch.cat ([h (x) for h in self.Concat], dim = -1))
    return output


###


---


### Definição de um Block (Pre-LayerNorm Transformer)

Em modelos GPT, um Block é:

- Layer Norm
- Attention
- Residual Connection (Add)
- Layer Norm
- Feed Forward Neural Network
- Residual Connection (Add)

In [12]:
class Block (nn.Module):
  def __init__(self):
    super().__init__()

    self.LayerNorm_Attention = nn.LayerNorm (Embedding_Dimension)
    self.Masked_Multi_Head_Attention = Multi_Head()


    self.FeedForward_NeuralNet = nn.Sequential (
        nn.Linear (Embedding_Dimension, 4 * Embedding_Dimension),
        nn.ReLU(),
        nn.Linear (Embedding_Dimension * 4, Embedding_Dimension)
    )
    self.LayerNorm_FNN = nn.LayerNorm (Embedding_Dimension)

  def forward (self, x):
    Norm_Attention = self.Masked_Multi_Head_Attention (self.LayerNorm_Attention (x)) #Norm #LayerNorm
    Add_Attention = x + Norm_Attention #Add #Residual Connection
    x = Add_Attention

    Norm_FNN = self.FeedForward_NeuralNet (self.LayerNorm_FNN (x)) #Norm #LayerNorm
    Add_FNN = x + Norm_FNN #Add #Residual Connection
    x = Add_FNN

    return x


###


---


### Modelo Adamastor-GPT

In [13]:
class ADAMASTOR_GPT (nn.Module):
  def __init__(self):
    super().__init__()
    self.Embedding = nn.Embedding (Tokenizer.get_vocab_size(), Embedding_Dimension)
    self.Positional_Encoding = nn.Embedding (Sequence_Length, Embedding_Dimension)

    self.Blocks = nn.Sequential (
        Block(),
        Block(),
        Block(),
        Block()
    )

    self.Language_Modeling = nn.Linear (Embedding_Dimension, Tokenizer.get_vocab_size()) #Converte o vetor final do modelo em logits sobre o vocabulário inteiro.

  def forward (self, x):
    B, T = x.shape

    x = self.Embedding (x) + self.Positional_Encoding (torch.arange (T, device = x.device))

    x = self.Blocks(x)

    logits = self.Language_Modeling (x)

    return logits #Return de um tensor (4, 8, 5000)



###


---


### Treino do Modelo (Super-Importante)


In [ ]:
def training (modelo, steps):

  modelo.train() #Ativa o treino
  optimizer = torch.optim.AdamW (modelo.parameters(), lr = 1e-4) #Usa-se AdamW ou SGD #Responsável por atualizar os pesos do modelo
  loss_function = nn.CrossEntropyLoss() #Função de perda que mede quão diferente está a previsão do modelo da resposta correta.

  for i in range (steps):

    x, y = Batch()

    logits = modelo (x)

    #print (logits.shape) #(16,32,5000)
    B, T, V = logits.shape #(Batch, Sequence Length, Vocab_Size)

    loss =  loss_function(
        logits.view (B * T, V),
        y.view (B * T)
    ) #Entrada que o Cross Entropy espera

    optimizer.zero_grad() #Zera os gradientes acumulados nos parâmetros.
    loss.backward() #Backpropagation

    optimizer.step() #Optimizer aplicado

    if i % 100 == 0: #Printa de 100 em 100
      print(f"Step {i} | Loss Value = {loss.item()} | PPL: {math.exp(loss.item())}")
      #Perplexity calcula quantas opções o modelo acha corretas para o próximo token, ou seja se tiver um perplexity baixo é bom, quer dizer que ele tem sempre a certeza do próximo token, o que significa que ele tem uma loss baixa. Podemos entender como uma maneira diferente de ver a loss
'''
    if i % 1000 == 0:
      print(gerar_texto(modelo, 100),"\n")
'''


###


---

### Geração de Texto

In [ ]:
def gerar_texto (modelo, max_tokens):

#É possível adicionar temperatura ao modelo, que no fundo é a creatividade dele

    device = next(modelo.parameters()).device  #Vai buscar o modelo à GPU
    tokens = torch.zeros((1, 1), dtype = torch.long, device = device)
    tokens = tokens.to(device)

    modelo.eval() #Desliga o treino

    with torch.no_grad(): #Desliga o gradiente
      for _ in range(max_tokens):

        contexto_max = tokens[:, -Sequence_Length:] #Alimento o modelo com o máximo de tokens possíveis, não posso dar um contexto maior do que daquele com que treinou

        logits = modelo(contexto_max) #(B,T,VOCAB_SIZE)
        logits = logits[:, -1, :] #O que vem depois do último token que gerou ?

        probabilidade = F.softmax(logits, dim = -1) #Aplica normalização

        next_token = torch.multinomial(probabilidade, num_samples = 1) #Seleciona um token aleatoriamente ponderado

        tokens = torch.cat([tokens, next_token], dim = 1) #Adiciona a uma lista com todos os outros tokens

      text = Tokenizer.decode(tokens[0].tolist()) #Decode do token para palavra ou sub palavra

      print(text)


###


---

### Adamastor-GPT Treino e Inferência

In [ ]:
modelo = ADAMASTOR_GPT().to(device)

training (modelo, 5000)

gerar_texto (modelo, 200)

In [ ]:
print (modelo.state_dict())
for param in modelo.state_dict():
  print (param)

In [ ]:
print (sum(p.numel() for p in modelo.parameters()))
save = torch.save(modelo.state_dict(), "Adamastor-GPT.pth")

O teu ADAMASTOR_GPT é:

um Transformer decoder-only (tipo GPT)
com:
4 camadas (blocos)
8 cabeças de atenção
embeddings de tamanho 64
vocabulário de 5000 tokens
contexto máximo de 32 tokens

FAZER ESTA LEITURA

interpretar loss value, o que o valor quer dizer ? <br>

CPU do PC vs TPU - tempo a treinar o modelo e resultados obtidos